# Notebook 05 – Training Pipeline

### Ziel dieses Notebooks

In diesem Notebook werden die finalen Trainingsruns für die drei Algorithmen PPO, A2C und DQN vorbereitet und durchgeführt.

Trainiert wird ausschließlich auf `Env1_Simple`. Die anderen beiden Environments (`Env2_Complex` und `Env3_Warehouse`) werden später in Notebook 06 zur Evaluation und Generalisierung verwendet.

Notebook 05 enthält bewusst keine finale Evaluation. Es geht nur um:

- Laden der finalen Trainingskonfiguration
- Laden der besten Optuna-Parameter für A2C und DQN
- Erstellen einer begründeten PPO-Konfiguration
- Training der finalen Modelle
- Speichern von Modellen, Logs und Run Summaries

Die eigentliche Bewertung der Algorithmen erfolgt erst in Notebook 06.

---

### Einordnung im Projekt

Bis zu diesem Punkt wurden die Unity-Environments aufgebaut, die Random Baseline ausgewertet, die Stable-Baselines3-Verbindung getestet und A2C sowie DQN mit Optuna getuned.

Die Optuna-Ergebnisse werden hier nicht als finale Leistungsbewertung verstanden, sondern nur als Grundlage für die Auswahl der finalen Trainingsparameter. Für PPO wird keine Optuna-Optimierung durchgeführt, da PPO direkt über Unity ML-Agents trainiert wird und dort über YAML-Dateien gesteuert wird.

Die finale Forschungslogik ist damit:

1. Training auf `Env1_Simple`
2. Evaluation auf `Env1_Simple`, `Env2_Complex` und `Env3_Warehouse`
3. Vergleich der Algorithmen anhand mehrerer Metriken

---

## Konstante Environment-Einstellungen

Für die Vergleichbarkeit mit der Random Baseline werden die zentralen Environment-Einstellungen nicht mehr verändert.

Konstant bleiben:

- Move Speed: `8`
- Turn Speed: `150`
- Max Step pro Episode: `5000`
- Decision Period: `1`
- Goal Reward: `+5`
- Wall Penalty: `-0.5`
- Step Penalty: `-0.001`
- Action Space: `Discrete(9)`
- Vector Observation Space Size: `4`
- Ray Perception Sensor
- Spawn- und Goalpunkt-Positionen
- Maze-Geometrie

Diese Werte wurden bereits in der Random Baseline verwendet. Würden sie jetzt verändert werden, wäre die Vergleichbarkeit zur Baseline nicht mehr sauber gegeben.

---

### Trainingssetup

Die finalen Trainingsruns verwenden dieselben fünf Seeds wie die Random Baseline:

`1, 27, 42, 72, 100`

Dadurch bleiben Random Baseline und trainierte Agenten auf Seed-Ebene vergleichbar.

Pro Algorithmus und Seed werden `500.000` Steps trainiert. Damit ist die finale Trainingsdauer zehnmal so groß wie die Optuna-Tuningdauer von `50.000` Steps pro Trial. Das soll den Agenten mehr Gelegenheit geben, eine stabilere Strategie zu lernen, ohne den zeitlichen Rahmen des Projekts komplett zu sprengen.

Insgesamt entstehen:

`3 Algorithmen × 5 Seeds = 15 finale Trainingsruns`

---

### Unity-Setup vor jedem SB3-Run

Für A2C und DQN muss Unity vor jedem Run vorbereitet sein:

- `Env1_Simple` ist geöffnet
- `MazeAgentScript` ist aktiv
- `RandomAgentScript` ist deaktiviert oder entfernt
- `EpisodeLogger` ist deaktiviert oder `enableLogging = false`
- Behavior Type = `Default`
- Behavior Name = `MazeAgent`
- Model = `None`
- Vector Observation Space Size = `4`
- Discrete Branches = `1`
- Branch 0 Size = `9`
- Decision Requester ist aktiv
- Decision Period = `1`
- Take Actions Between Decisions ist aktiviert
- Max Step = `5000`

Wichtig: Bei jeder Unity-Verbindung wird zuerst die Python-Zelle gestartet und danach in Unity auf Play gedrückt.

---

### Imports

In dieser Zelle werden alle Bibliotheken geladen, die für die finale Trainingspipeline benötigt werden. Zusätzlich wird der Projektroot gesetzt, damit Pfade sauber relativ zum Projektordner bleiben.

In [17]:
from pathlib import Path
import sys
import os
import json
import random
import time
from tqdm.auto import tqdm
from datetime import datetime
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces

PROJECT_ROOT_NOTEBOOK = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT_NOTEBOOK))

from lib_py.paths import (
    PROJECT_ROOT,
    PROJECT_NAME,
    MAZE_AGENT_DIR,
    TRAINING_DIR,
    MODEL_DIR,
    LOG_DIR,
    ensure_project_dirs,
    show_project_path,
)

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

from stable_baselines3 import A2C, DQN
from stable_baselines3.common.callbacks import BaseCallback

os.chdir(PROJECT_ROOT)

print("Imports funktionieren.")
print("Project Root:", PROJECT_NAME)

Imports funktionieren.
Project Root: rl_navigation_project


---

### Projektordner vorbereiten

In dieser Zelle werden die zentralen Output-Ordner für die finalen Trainingsruns erstellt. Die finalen Outputs werden bewusst getrennt von Wrapper-Tests und Optuna-Logs gespeichert.

In [3]:
ensure_project_dirs()

CONFIG_DIR = TRAINING_DIR / "configs"
TUNING_DIR = TRAINING_DIR / "tuning"
FINAL_MODEL_DIR = MODEL_DIR / "final"
FINAL_LOG_DIR = LOG_DIR / "final"
RUN_SUMMARY_DIR = TRAINING_DIR / "run_summaries"
RECORDING_DIR = TRAINING_DIR / "recordings"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TUNING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
RECORDING_DIR.mkdir(parents=True, exist_ok=True)

print("Projektordner vorbereitet.")
print("Config Dir:", show_project_path(CONFIG_DIR))
print("Tuning Dir:", show_project_path(TUNING_DIR))
print("Final Model Dir:", show_project_path(FINAL_MODEL_DIR))
print("Final Log Dir:", show_project_path(FINAL_LOG_DIR))
print("Run Summary Dir:", show_project_path(RUN_SUMMARY_DIR))
print("Recording Dir:", show_project_path(RECORDING_DIR))

Projektordner vorbereitet.
Config Dir: rl_navigation_project/training/configs
Tuning Dir: rl_navigation_project/training/tuning
Final Model Dir: rl_navigation_project/training/models/final
Final Log Dir: rl_navigation_project/training/logs/final
Run Summary Dir: rl_navigation_project/training/run_summaries
Recording Dir: rl_navigation_project/training/recordings


---

### Finale Trainingskonfiguration

Die finalen Runs trainieren auf `Env1_Simple` mit fünf Seeds. Die Schrittzahl pro Run beträgt `500.000`.

Die Seeds entsprechen bewusst der Random Baseline, damit die Ergebnisse später konsistent miteinander verglichen werden können.

In [4]:
ENV_NAME = "Env1"
UNITY_ENV_NAME = "Env1_Simple"

FINAL_TIMESTEPS = 500_000
CHUNK_TIMESTEPS = 100_000
TRAINING_SEEDS = [1, 27, 42, 72, 100]

print("Finale Trainingskonfiguration gesetzt.")
print("Training Environment:", UNITY_ENV_NAME)
print("Final Timesteps pro Run:", FINAL_TIMESTEPS)
print("Chunk Timesteps:", CHUNK_TIMESTEPS)
print("Training Seeds:", TRAINING_SEEDS)

Finale Trainingskonfiguration gesetzt.
Training Environment: Env1_Simple
Final Timesteps pro Run: 500000
Chunk Timesteps: 100000
Training Seeds: [1, 27, 42, 72, 100]


---

### Beste Parameter aus Notebook 04 laden

A2C und DQN verwenden die besten getesteten Parameterkombinationen aus dem Optuna-Tuning. Die Werte werden aus JSON-Dateien geladen und nicht hardcoded, damit Notebook 05 sauber auf Notebook 04 aufbaut.

Die Parameter sind nicht als global optimale Hyperparameter zu verstehen, sondern als beste getestete Kombination innerhalb des begrenzten Optuna-Suchraums.

In [5]:
a2c_params_path = TUNING_DIR / "best_params_a2c.json"
dqn_params_path = TUNING_DIR / "best_params_dqn.json"

with open(a2c_params_path, "r", encoding="utf-8") as file:
    BEST_PARAMS_A2C = json.load(file)

with open(dqn_params_path, "r", encoding="utf-8") as file:
    BEST_PARAMS_DQN = json.load(file)

print("A2C Best Params geladen aus:", show_project_path(a2c_params_path))
print(BEST_PARAMS_A2C)

print("\nDQN Best Params geladen aus:", show_project_path(dqn_params_path))
print(BEST_PARAMS_DQN)

A2C Best Params geladen aus: rl_navigation_project/training/tuning/best_params_a2c.json
{'learning_rate': 1.1973220476107518e-05, 'n_steps': 5, 'gamma': 0.995, 'ent_coef': 0.002597549498493029, 'vf_coef': 0.75, 'max_grad_norm': 0.5}

DQN Best Params geladen aus: rl_navigation_project/training/tuning/best_params_dqn.json
{'learning_rate': 4.9228176440041644e-05, 'buffer_size': 100000, 'learning_starts': 5000, 'batch_size': 32, 'gamma': 0.995, 'exploration_fraction': 0.3, 'exploration_final_eps': 0.02, 'target_update_interval': 1000}


**Begründung der A2C- und DQN-Konfiguration**

A2C und DQN starten mit den besten getesteten Parameterkombinationen aus dem Optuna-Tuning. Dabei ist wichtig, dass Optuna in diesem Projekt nur als kompakter Suchlauf genutzt wurde. Es wurde keine vollständige Hyperparameter-Optimierung durchgeführt, sondern eine begrenzte Parameterauswahl innerhalb eines kleinen Suchraums.

Deshalb werden die Optuna-Parameter zunächst als Ausgangskonfiguration für die finalen Trainingsruns verwendet. Für beide Algorithmen wird zuerst Seed 1 trainiert und anschließend über TensorBoard geprüft, ob der Trainingsverlauf grundsätzlich plausibel ist. Falls ein Algorithmus nach diesem ersten längeren Run weiterhin keinerlei Lernfortschritt zeigt, werden die Hyperparameter vor den restlichen Seeds manuell überprüft und gegebenenfalls angepasst.

Wichtig ist dabei jedoch zu erwähnen, dass jede nachträgliche Anpassung dokumentiert wird. Die finale Trainingskonfiguration muss für alle Seeds eines Algorithmus konsistent sein, damit die späteren Evaluationsergebnisse vergleichbar bleiben.

Daraus folgt ebenfalls, dass das folgende Prinzip gilt (sollte der erste Run mit Seed 1 angepasst werden müssen):
- Seed 1 alter Run = Pilot-/Reality-Check-Run
- Seed 1 neuer Run = finaler Run mit finaler Konfiguration
- Seeds 27, 42, 72, 100 = gleiche finale Konfiguration

Ansonsten wäre es so dass ein Seed mit anderen Hyperparametern trainert wurde als die anderen.

---

### Begründung der PPO-Hyperparameter

Die PPO-Konfiguration ist bewusst stabil und nicht zu aggressiv gewählt.

- `learning_rate = 0.0003`: typischer Startwert für PPO und nicht zu aggressiv.
- `batch_size = 128`: ausreichend groß für stabile Updates, aber noch kompakt genug für das Projekt.
- `buffer_size = 2048`: PPO sammelt mehrere Erfahrungen, bevor ein Update durchgeführt wird.
- `gamma = 0.995`: hoher Discount Factor, weil Navigation oft langfristige Rewards hat.
- `beta = 0.005`: Entropie-Regularisierung, damit Exploration erhalten bleibt.
- `epsilon = 0.2`: PPO-Clipping zur Stabilisierung der Policy-Updates.
- `lambd = 0.95`: GAE-Parameter zur Balance zwischen Bias und Varianz.
- `num_epoch = 3`: mehrere Updates pro Batch, ohne zu stark zu überfitten.
- `hidden_units = 128`, `num_layers = 2`: ausreichend großes Netzwerk für die Vektor- und Ray-Observations, aber nicht unnötig komplex.
- `time_horizon = 64`: begrenzter Erfahrungshorizont pro Update, passend für PPO in ML-Agents.
- `max_steps = 500000`: identisch zur finalen Trainingsdauer der SB3-Algorithmen.

Die Konfiguration ist damit als solide PPO-Baseline für Unity ML-Agents zu verstehen und nicht als maximal optimierte PPO-Konfiguration.

### PPO YAML-Datei erstellen

In dieser Zelle wird die finale PPO-Konfiguration als YAML-Datei gespeichert. Diese Datei wird später im Terminal mit `mlagents-learn` verwendet.

In [6]:
ppo_config_text = f"""
behaviors:
  MazeAgent:
    trainer_type: ppo

    hyperparameters:
      batch_size: 128
      buffer_size: 2048
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear

    network_settings:
      normalize: true
      hidden_units: 128
      num_layers: 2

    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0

    max_steps: {FINAL_TIMESTEPS}
    time_horizon: 64
    summary_freq: 10000
"""

ppo_config_path = CONFIG_DIR / "ppo_env1_config.yaml"

with open(ppo_config_path, "w", encoding="utf-8") as file:
    file.write(ppo_config_text.strip() + "\n")

print("PPO Config gespeichert:", show_project_path(ppo_config_path))

PPO Config gespeichert: rl_navigation_project/training/configs/ppo_env1_config.yaml


---

### PPO Terminal-Befehle

PPO wird nicht direkt aus dem Notebook trainiert. Stattdessen werden die folgenden Befehle im Terminal ausgeführt. Für jeden Seed wird eine eigene Run-ID verwendet.

Nach Start des jeweiligen Befehls muss Unity in den Play Mode gesetzt werden.

In [7]:
ppo_commands = []

for seed in TRAINING_SEEDS:
    run_id = f"PPO_{ENV_NAME}_Seed{seed}"
    command = (
        f"mlagents-learn {show_project_path(ppo_config_path)} "
        f"--run-id={run_id} "
        f"--seed={seed} "
        f"--results-dir=training/logs/final"
    )
    ppo_commands.append({"algorithm": "PPO", "seed": seed, "run_id": run_id, "command": command})

ppo_commands_df = pd.DataFrame(ppo_commands)

ppo_commands_path = RUN_SUMMARY_DIR / "ppo_training_commands.csv"
ppo_commands_df.to_csv(ppo_commands_path, index=False)

print("PPO Commands gespeichert:", show_project_path(ppo_commands_path))

ppo_commands_df

PPO Commands gespeichert: rl_navigation_project/training/run_summaries/ppo_training_commands.csv


,algorithm,seed,run_id,command
0,PPO,1,PPO_Env1_Seed1,mlagents-learn rl_navigation_project/training/...
1,PPO,27,PPO_Env1_Seed27,mlagents-learn rl_navigation_project/training/...
2,PPO,42,PPO_Env1_Seed42,mlagents-learn rl_navigation_project/training/...
3,PPO,72,PPO_Env1_Seed72,mlagents-learn rl_navigation_project/training/...
4,PPO,100,PPO_Env1_Seed100,mlagents-learn rl_navigation_project/training/...


---

### Seeds setzen

Für A2C und DQN werden die wichtigsten Python-Seeds gesetzt. Zusätzlich wird der Seed an Stable-Baselines3 und an die Unity-Verbindung übergeben. In Unity sollte der `SeedManager` passend zum jeweiligen Run eingestellt sein.

In [9]:
def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"Seeds gesetzt auf: {seed}")

---

### UnityFlatGymnasiumWrapper

A2C und DQN werden über Stable-Baselines3 trainiert. Dafür wird derselbe Wrapper genutzt, der bereits in Notebook 03 entwickelt und getestet wurde.

Der Wrapper führt mehrere Unity-Observations zu einem flachen Vektor zusammen und stellt das Environment im Gymnasium-Format bereit.

In [10]:
class UnityFlatGymnasiumWrapper(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, unity_gym_env):
        super().__init__()

        self.unity_gym_env = unity_gym_env
        self.action_space = spaces.Discrete(self.unity_gym_env.action_space.n)

        initial_obs = self.unity_gym_env.reset()
        flat_obs = self._flatten_obs(initial_obs)

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=flat_obs.shape,
            dtype=np.float32
        )

    def _flatten_obs(self, obs: Any) -> np.ndarray:
        if isinstance(obs, tuple) and len(obs) == 2 and isinstance(obs[1], dict):
            obs = obs[0]

        if isinstance(obs, dict):
            parts = [np.asarray(value, dtype=np.float32).ravel() for value in obs.values()]
            return np.concatenate(parts).astype(np.float32)

        if isinstance(obs, (list, tuple)):
            parts = [np.asarray(part, dtype=np.float32).ravel() for part in obs]
            return np.concatenate(parts).astype(np.float32)

        return np.asarray(obs, dtype=np.float32).ravel()

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            np.random.seed(seed)

        obs = self.unity_gym_env.reset()
        flat_obs = self._flatten_obs(obs)

        return flat_obs, {}

    def step(self, action):
        action = int(action)

        step_result = self.unity_gym_env.step(action)

        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result
        else:
            obs, reward, done, info = step_result
            terminated = bool(done)
            truncated = False

        flat_obs = self._flatten_obs(obs)

        return flat_obs, float(reward), terminated, truncated, info

    def close(self):
        self.unity_gym_env.close()

---

### Unity-Environment erstellen

Diese Funktion verbindet Python mit dem Unity Editor. Pro Trainingsrun wird eine neue Verbindung aufgebaut. Dadurch bleiben die Runs sauber getrennt.

In [11]:
def make_unity_env(seed: int) -> UnityFlatGymnasiumWrapper:
    print("Starte Verbindung zu Unity. Jetzt in Unity Play drücken...")

    unity_env = UnityEnvironment(
        file_name=None,
        seed=seed,
        side_channels=[]
    )

    unity_gym_env = UnityToGymWrapper(
        unity_env,
        uint8_visual=False,
        flatten_branched=True,
        allow_multiple_obs=True
    )

    wrapped_env = UnityFlatGymnasiumWrapper(unity_gym_env)

    print("Unity Environment verbunden.")
    print("Observation Space:", wrapped_env.observation_space)
    print("Action Space:", wrapped_env.action_space)

    return wrapped_env

---

### Training-Callback

Der folgende Callback sammelt Rewards während des Trainings. Das ist keine finale Evaluation, sondern nur ein einfacher Trainingsindikator. Er hilft dabei, nach einem Run zu sehen, ob sich die letzten Trainingsepisoden verbessert haben.

Die finale Bewertung erfolgt trotzdem erst in Notebook 06.

In [12]:
class TrainingRewardCallback(BaseCallback):
    def __init__(self, print_every_steps: int = 100_000, verbose: int = 0):
        super().__init__(verbose)
        self.print_every_steps = print_every_steps
        self.current_episode_reward = 0.0
        self.episode_rewards = []
        self.last_print_step = 0

    def _on_step(self) -> bool:
        rewards = self.locals.get("rewards")
        dones = self.locals.get("dones")

        if rewards is not None:
            self.current_episode_reward += float(np.asarray(rewards).mean())

        if dones is not None and bool(np.asarray(dones).any()):
            self.episode_rewards.append(self.current_episode_reward)
            self.current_episode_reward = 0.0

        if self.num_timesteps - self.last_print_step >= self.print_every_steps:
            recent_mean = self.get_recent_mean_reward()
            print(
                f"Training Step {self.num_timesteps}: "
                f"recent_training_mean_reward={recent_mean}"
            )
            self.last_print_step = self.num_timesteps

        return True

    def get_recent_mean_reward(self, last_n: int = 20):
        if len(self.episode_rewards) == 0:
            return None

        return float(np.mean(self.episode_rewards[-last_n:]))

---

### Run Summary speichern

Nach jedem Run wird eine JSON-Datei gespeichert. Sie enthält die wichtigsten Informationen zum Training und macht den Run später nachvollziehbar.

In [13]:
def save_run_summary(summary: dict) -> Path:
    summary_path = RUN_SUMMARY_DIR / f"{summary['run_id']}_summary.json"

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)

    print("Run Summary gespeichert:", show_project_path(summary_path))

    return summary_path

---

### A2C final trainieren

A2C wird mit den besten getesteten Optuna-Parametern trainiert. Jeder Run nutzt einen Seed und wird als eigenes Modell gespeichert.

Das Training läuft automatisch bis `500.000` Steps durch. Intern wird es nur in Blöcke von `100.000` Steps aufgeteilt, damit nach jedem Block ein kurzer Zwischenstand ausgegeben wird. Unity muss dabei nicht neu gestartet werden.

Für jeden Seed wird eine eigene Zelle verwendet, damit die Runs bewusst einzeln gestartet werden können.

In [16]:
def train_a2c_final(seed: int) -> dict:
    set_global_seeds(seed)

    run_id = f"A2C_{ENV_NAME}_Seed{seed}"
    model_dir = FINAL_MODEL_DIR / run_id
    log_dir_rel = Path("training") / "logs" / "final" / run_id
    log_dir_abs = PROJECT_ROOT / log_dir_rel

    model_dir.mkdir(parents=True, exist_ok=True)
    log_dir_abs.mkdir(parents=True, exist_ok=True)

    start_time = time.time()
    started_at = datetime.now().isoformat(timespec="seconds")

    env = None

    try:
        env = make_unity_env(seed)

        model = A2C(
            policy="MlpPolicy",
            env=env,
            learning_rate=BEST_PARAMS_A2C["learning_rate"],
            n_steps=BEST_PARAMS_A2C["n_steps"],
            gamma=BEST_PARAMS_A2C["gamma"],
            ent_coef=BEST_PARAMS_A2C["ent_coef"],
            vf_coef=BEST_PARAMS_A2C["vf_coef"],
            max_grad_norm=BEST_PARAMS_A2C["max_grad_norm"],
            seed=seed,
            verbose=0,
            tensorboard_log=str(log_dir_rel)
        )

        callback = TrainingRewardCallback(print_every_steps=100_000)

        trained_steps = 0
        total_chunks = int(np.ceil(FINAL_TIMESTEPS / CHUNK_TIMESTEPS))

        with tqdm(total=FINAL_TIMESTEPS, desc=run_id, unit="steps") as progress:
            for chunk_idx in range(1, total_chunks + 1):
                remaining_steps = FINAL_TIMESTEPS - trained_steps
                current_chunk_steps = min(CHUNK_TIMESTEPS, remaining_steps)

                print(f"{run_id}: Starte Chunk {chunk_idx}/{total_chunks} mit {current_chunk_steps} Steps.")

                model.learn(
                    total_timesteps=current_chunk_steps,
                    tb_log_name=run_id,
                    reset_num_timesteps=(trained_steps == 0),
                    callback=callback,
                    progress_bar=False
                )

                trained_steps += current_chunk_steps
                progress.update(current_chunk_steps)

                recent_reward = callback.get_recent_mean_reward()
                print(
                    f"{run_id}: Chunk {chunk_idx}/{total_chunks} abgeschlossen. "
                    f"Trained Steps: {trained_steps}. "
                    f"Recent Training Mean Reward: {recent_reward}"
                )

                if trained_steps >= 200_000 and recent_reward is not None and recent_reward <= -1.0:
                    print(
                        "Hinweis: Nach mindestens 200.000 Steps liegt der recent_training_mean_reward "
                        "weiterhin bei etwa -1.0 oder schlechter. Bitte TensorBoard prüfen, bevor weitere Seeds gestartet werden."
                    )

        model_path = model_dir / "final_model"
        model.save(model_path)

        duration_seconds = round(time.time() - start_time, 2)
        finished_at = datetime.now().isoformat(timespec="seconds")

        summary = {
            "run_id": run_id,
            "algorithm": "A2C",
            "environment": UNITY_ENV_NAME,
            "seed": seed,
            "timesteps": FINAL_TIMESTEPS,
            "trained_steps": trained_steps,
            "hyperparameters": BEST_PARAMS_A2C,
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_seconds": duration_seconds,
            "recent_training_mean_reward": callback.get_recent_mean_reward(),
            "model_path": show_project_path(model_path.with_suffix(".zip")),
            "log_dir": show_project_path(log_dir_abs),
            "recording_dir": show_project_path(RECORDING_DIR),
        }

        save_run_summary(summary)

        print(f"{run_id}: Training abgeschlossen.")
        print("Model:", summary["model_path"])
        print("Logs:", summary["log_dir"])

        return summary

    finally:
        if env is not None:
            env.close()
            print(f"{run_id}: Unity Environment geschlossen.")

---

## A2C Trainingsläufe

#### A2C Seed 1 trainieren

In dieser Zelle wird der erste finale A2C-Run gestartet. Dieser Run dient gleichzeitig als Reality Check. Nach Abschluss sollte TensorBoard geprüft werden, bevor die weiteren A2C-Seeds gestartet werden.

In [18]:
a2c_seed1_summary = train_a2c_final(seed=1)
a2c_seed1_summary

Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)


A2C_Env1_Seed1:   0%|          | 0/500000 [00:00<?, ?steps/s]

A2C_Env1_Seed1: Starte Chunk 1/5 mit 100000 Steps.
Training Step 100000: recent_training_mean_reward=-1.1279500169039238
A2C_Env1_Seed1: Chunk 1/5 abgeschlossen. Trained Steps: 100000. Recent Training Mean Reward: -1.1279500169039238
A2C_Env1_Seed1: Starte Chunk 2/5 mit 100000 Steps.
Training Step 200000: recent_training_mean_reward=-1.9948000599106308
A2C_Env1_Seed1: Chunk 2/5 abgeschlossen. Trained Steps: 200000. Recent Training Mean Reward: -1.9948000599106308
Hinweis: Nach mindestens 200.000 Steps liegt der recent_training_mean_reward weiterhin bei etwa -1.0 oder schlechter. Bitte TensorBoard prüfen, bevor weitere Seeds gestartet werden.
A2C_Env1_Seed1: Starte Chunk 3/5 mit 100000 Steps.
Training Step 300000: recent_training_mean_reward=-1.2797500360291452
A2C_Env1_Seed1: Chunk 3/5 abgeschlossen. Trained Steps: 300000. Recent Training Mean Reward: -1.2797500360291452
Hinweis: Nach mindestens 200.000 Steps liegt der recent_training_mean_reward weiterhin bei etwa -1.0 oder schlechter

{'run_id': 'A2C_Env1_Seed1',
 'algorithm': 'A2C',
 'environment': 'Env1_Simple',
 'seed': 1,
 'timesteps': 500000,
 'trained_steps': 500000,
 'hyperparameters': {'learning_rate': 1.1973220476107518e-05,
  'n_steps': 5,
  'gamma': 0.995,
  'ent_coef': 0.002597549498493029,
  'vf_coef': 0.75,
  'max_grad_norm': 0.5},
 'started_at': '2026-05-07T22:24:42',
 'finished_at': '2026-05-08T01:12:27',
 'duration_seconds': 10065.12,
 'recent_training_mean_reward': -2.833850108931074,
 'model_path': 'rl_navigation_project/training/models/final/A2C_Env1_Seed1/final_model.zip',
 'log_dir': 'rl_navigation_project/training/logs/final/A2C_Env1_Seed1',
 'recording_dir': 'rl_navigation_project/training/recordings'}

**Sehr schlechtes Ergebnis, muss manuelle Anpassung vornehmen - Optuna war nicht gut genug (verständlicherweise)**

#### A2C Seed 27 trainieren

Diese Zelle startet den finalen A2C-Run mit Seed 27. Die Konfiguration bleibt identisch zu Seed 1, nur der Seed wird geändert.

In [ ]:
a2c_seed27_summary = train_a2c_final(seed=27)
a2c_seed27_summary

#### A2C Seed 42 trainieren

Diese Zelle startet den finalen A2C-Run mit Seed 42. Auch hier bleibt die Trainingskonfiguration identisch, damit die Seeds später fair miteinander verglichen werden können.

In [ ]:
a2c_seed42_summary = train_a2c_final(seed=42)
a2c_seed42_summary

#### A2C Seed 72 trainieren

Diese Zelle startet den finalen A2C-Run mit Seed 72.

In [ ]:
a2c_seed72_summary = train_a2c_final(seed=72)
a2c_seed72_summary

#### A2C Seed 100 trainieren

Diese Zelle startet den finalen A2C-Run mit Seed 100.

In [ ]:
a2c_seed100_summary = train_a2c_final(seed=100)
a2c_seed100_summary

#### A2C Run Summaries sammeln

Diese Zelle sammelt alle A2C-Runs, die im aktuellen Notebook-Kernel bereits abgeschlossen wurden. Dadurch kann die Zelle auch ausgeführt werden, wenn bisher nur einzelne Seeds trainiert wurden.

In [ ]:
a2c_summary_variable_names = [
    "a2c_seed1_summary",
    "a2c_seed27_summary",
    "a2c_seed42_summary",
    "a2c_seed72_summary",
    "a2c_seed100_summary",
]

a2c_available_summaries = [
    globals()[name]
    for name in a2c_summary_variable_names
    if name in globals()
]

if len(a2c_available_summaries) > 0:
    a2c_runs_df = pd.DataFrame(a2c_available_summaries)

    a2c_runs_path = RUN_SUMMARY_DIR / "a2c_final_training_runs.csv"
    a2c_runs_df.to_csv(a2c_runs_path, index=False)

    print("A2C Run Summary CSV gespeichert:", show_project_path(a2c_runs_path))

    display(a2c_runs_df)
else:
    print("Es wurden in diesem Kernel noch keine A2C-Runs abgeschlossen.")

---

### DQN final trainieren

DQN wird mit den besten getesteten Optuna-Parametern trainiert. Jeder Run nutzt einen Seed und wird als eigenes Modell gespeichert.

Wie bei A2C läuft das Training automatisch bis `500.000` Steps durch und wird intern nur in `100.000`er-Blöcke aufgeteilt, damit Zwischenstände sichtbar werden. Unity muss während eines Runs nicht neu gestartet werden.

Auch DQN wird bewusst pro Seed in einer eigenen Zelle gestartet.

In [ ]:
def train_dqn_final(seed: int) -> dict:
    set_global_seeds(seed)

    run_id = f"DQN_{ENV_NAME}_Seed{seed}"
    model_dir = FINAL_MODEL_DIR / run_id
    log_dir_rel = Path("training") / "logs" / "final" / run_id
    log_dir_abs = PROJECT_ROOT / log_dir_rel

    model_dir.mkdir(parents=True, exist_ok=True)
    log_dir_abs.mkdir(parents=True, exist_ok=True)

    start_time = time.time()
    started_at = datetime.now().isoformat(timespec="seconds")

    env = None

    try:
        env = make_unity_env(seed)

        model = DQN(
            policy="MlpPolicy",
            env=env,
            learning_rate=BEST_PARAMS_DQN["learning_rate"],
            buffer_size=BEST_PARAMS_DQN["buffer_size"],
            learning_starts=BEST_PARAMS_DQN["learning_starts"],
            batch_size=BEST_PARAMS_DQN["batch_size"],
            gamma=BEST_PARAMS_DQN["gamma"],
            exploration_fraction=BEST_PARAMS_DQN["exploration_fraction"],
            exploration_initial_eps=1.0,
            exploration_final_eps=BEST_PARAMS_DQN["exploration_final_eps"],
            target_update_interval=BEST_PARAMS_DQN["target_update_interval"],
            seed=seed,
            verbose=0,
            tensorboard_log=str(log_dir_rel)
        )

        callback = TrainingRewardCallback(print_every_steps=100_000)

        trained_steps = 0
        total_chunks = int(np.ceil(FINAL_TIMESTEPS / CHUNK_TIMESTEPS))

        with tqdm(total=FINAL_TIMESTEPS, desc=run_id, unit="steps") as progress:
            for chunk_idx in range(1, total_chunks + 1):
                remaining_steps = FINAL_TIMESTEPS - trained_steps
                current_chunk_steps = min(CHUNK_TIMESTEPS, remaining_steps)

                print(f"{run_id}: Starte Chunk {chunk_idx}/{total_chunks} mit {current_chunk_steps} Steps.")

                model.learn(
                    total_timesteps=current_chunk_steps,
                    tb_log_name=run_id,
                    reset_num_timesteps=(trained_steps == 0),
                    callback=callback,
                    progress_bar=False
                )

                trained_steps += current_chunk_steps
                progress.update(current_chunk_steps)

                recent_reward = callback.get_recent_mean_reward()
                print(
                    f"{run_id}: Chunk {chunk_idx}/{total_chunks} abgeschlossen. "
                    f"Trained Steps: {trained_steps}. "
                    f"Recent Training Mean Reward: {recent_reward}"
                )

                if trained_steps >= 200_000 and recent_reward is not None and recent_reward <= -1.0:
                    print(
                        "Hinweis: Nach mindestens 200.000 Steps liegt der recent_training_mean_reward "
                        "weiterhin bei etwa -1.0 oder schlechter. Bitte TensorBoard prüfen, bevor weitere Seeds gestartet werden."
                    )

        model_path = model_dir / "final_model"
        model.save(model_path)

        duration_seconds = round(time.time() - start_time, 2)
        finished_at = datetime.now().isoformat(timespec="seconds")

        summary = {
            "run_id": run_id,
            "algorithm": "DQN",
            "environment": UNITY_ENV_NAME,
            "seed": seed,
            "timesteps": FINAL_TIMESTEPS,
            "trained_steps": trained_steps,
            "hyperparameters": BEST_PARAMS_DQN,
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_seconds": duration_seconds,
            "recent_training_mean_reward": callback.get_recent_mean_reward(),
            "model_path": show_project_path(model_path.with_suffix(".zip")),
            "log_dir": show_project_path(log_dir_abs),
            "recording_dir": show_project_path(RECORDING_DIR),
        }

        save_run_summary(summary)

        print(f"{run_id}: Training abgeschlossen.")
        print("Model:", summary["model_path"])
        print("Logs:", summary["log_dir"])

        return summary

    finally:
        if env is not None:
            env.close()
            print(f"{run_id}: Unity Environment geschlossen.")

---

## DQN Trainingsläufe

#### DQN Seed 1 trainieren

In dieser Zelle wird der erste finale DQN-Run gestartet. Dieser Run dient ebenfalls als Reality Check. Nach Abschluss sollte TensorBoard geprüft werden, bevor die weiteren DQN-Seeds gestartet werden.

In [ ]:
dqn_seed1_summary = train_dqn_final(seed=1)
dqn_seed1_summary

#### DQN Seed 27 trainieren

Diese Zelle startet den finalen DQN-Run mit Seed 27. Die Konfiguration bleibt identisch zu Seed 1.

In [ ]:
dqn_seed27_summary = train_dqn_final(seed=27)
dqn_seed27_summary

#### DQN Seed 42 trainieren

Diese Zelle startet den finalen DQN-Run mit Seed 42.

In [ ]:
dqn_seed42_summary = train_dqn_final(seed=42)
dqn_seed42_summary

#### DQN Seed 72 trainieren

Diese Zelle startet den finalen DQN-Run mit Seed 72.

In [ ]:
dqn_seed72_summary = train_dqn_final(seed=72)
dqn_seed72_summary

#### DQN Seed 100 trainieren

Diese Zelle startet den finalen DQN-Run mit Seed 100.

In [ ]:
dqn_seed100_summary = train_dqn_final(seed=100)
dqn_seed100_summary

#### DQN Run Summaries sammeln

Diese Zelle sammelt alle DQN-Runs, die im aktuellen Notebook-Kernel bereits abgeschlossen wurden. Dadurch kann die Zelle auch nach einzelnen Seeds schon eine Zwischenübersicht speichern.

In [ ]:
dqn_summary_variable_names = [
    "dqn_seed1_summary",
    "dqn_seed27_summary",
    "dqn_seed42_summary",
    "dqn_seed72_summary",
    "dqn_seed100_summary",
]

dqn_available_summaries = [
    globals()[name]
    for name in dqn_summary_variable_names
    if name in globals()
]

if len(dqn_available_summaries) > 0:
    dqn_runs_df = pd.DataFrame(dqn_available_summaries)

    dqn_runs_path = RUN_SUMMARY_DIR / "dqn_final_training_runs.csv"
    dqn_runs_df.to_csv(dqn_runs_path, index=False)

    print("DQN Run Summary CSV gespeichert:", show_project_path(dqn_runs_path))

    display(dqn_runs_df)
else:
    print("Es wurden in diesem Kernel noch keine DQN-Runs abgeschlossen.")

---

### PPO final trainieren

PPO wird über Unity ML-Agents trainiert und deshalb nicht direkt über Stable-Baselines3 im Notebook ausgeführt. Das Notebook erstellt die PPO-Konfiguration und dokumentiert die Terminal-Befehle für jeden Seed.

Für jeden PPO-Run wird im Terminal ein eigener `mlagents-learn`-Befehl gestartet. Danach muss Unity in den Play Mode gesetzt werden.

#### PPO Seed 1 trainieren

PPO wird für Seed 1 mit folgendem Terminal-Befehl gestartet:

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed1 --seed=1 --results-dir=training/logs/final
```

Nach Start des Befehls muss Unity in den Play Mode gesetzt werden.

#### PPO Seed 27 trainieren

PPO wird für Seed 27 mit folgendem Terminal-Befehl gestartet:

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed27 --seed=27 --results-dir=training/logs/final
```

Nach Start des Befehls muss Unity in den Play Mode gesetzt werden.

#### PPO Seed 42 trainieren

PPO wird für Seed 42 mit folgendem Terminal-Befehl gestartet:

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed42 --seed=42 --results-dir=training/logs/final
```

Nach Start des Befehls muss Unity in den Play Mode gesetzt werden.

#### PPO Seed 72 trainieren

PPO wird für Seed 72 mit folgendem Terminal-Befehl gestartet:

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed72 --seed=72 --results-dir=training/logs/final
```

Nach Start des Befehls muss Unity in den Play Mode gesetzt werden.

#### PPO Seed 100 trainieren

PPO wird für Seed 100 mit folgendem Terminal-Befehl gestartet:

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed100 --seed=100 --results-dir=training/logs/final
```

Nach Start des Befehls muss Unity in den Play Mode gesetzt werden.

---

#### PPO Run Summary vorbereiten

Da PPO über das Terminal trainiert wird, erzeugt dieses Notebook zunächst eine geplante PPO-Run-Übersicht. Nach den PPO-Runs können die tatsächlichen Laufzeiten und Modellpfade bei Bedarf ergänzt werden.

In [ ]:
ppo_planned_summaries = []

for seed in TRAINING_SEEDS:
    run_id = f"PPO_{ENV_NAME}_Seed{seed}"
    expected_log_dir = FINAL_LOG_DIR / run_id
    expected_model_path = expected_log_dir / "MazeAgent.onnx"

    summary = {
        "run_id": run_id,
        "algorithm": "PPO",
        "environment": UNITY_ENV_NAME,
        "seed": seed,
        "timesteps": FINAL_TIMESTEPS,
        "trained_steps": None,
        "hyperparameters": {
            "trainer_type": "ppo",
            "batch_size": 128,
            "buffer_size": 2048,
            "learning_rate": 0.0003,
            "beta": 0.005,
            "epsilon": 0.2,
            "lambd": 0.95,
            "num_epoch": 3,
            "gamma": 0.995,
            "hidden_units": 128,
            "num_layers": 2,
            "time_horizon": 64,
            "max_steps": FINAL_TIMESTEPS,
        },
        "started_at": None,
        "finished_at": None,
        "duration_seconds": None,
        "recent_training_mean_reward": None,
        "model_path": show_project_path(expected_model_path),
        "log_dir": show_project_path(expected_log_dir),
        "recording_dir": show_project_path(RECORDING_DIR),
    }

    ppo_planned_summaries.append(summary)

ppo_planned_df = pd.DataFrame(ppo_planned_summaries)

ppo_planned_path = RUN_SUMMARY_DIR / "ppo_planned_training_runs.csv"
ppo_planned_df.to_csv(ppo_planned_path, index=False)

print("Geplante PPO Run Summary gespeichert:", show_project_path(ppo_planned_path))

ppo_planned_df

---

### SB3-Runs zusammenführen

In dieser Zelle werden die bereits vorhandenen A2C- und DQN-Run-Summaries zusammengeführt. Die Zelle funktioniert auch dann, wenn bisher nur A2C oder nur DQN trainiert wurde.

In [ ]:
available_frames = []

if "a2c_runs_df" in globals():
    available_frames.append(a2c_runs_df)

if "dqn_runs_df" in globals():
    available_frames.append(dqn_runs_df)

if len(available_frames) > 0:
    sb3_runs_df = pd.concat(available_frames, ignore_index=True)

    sb3_runs_path = RUN_SUMMARY_DIR / "sb3_final_training_runs.csv"
    sb3_runs_df.to_csv(sb3_runs_path, index=False)

    print("SB3 Run Summary gespeichert:", show_project_path(sb3_runs_path))

    display(sb3_runs_df)
else:
    print("Noch keine A2C/DQN-Runs in diesem Kernel abgeschlossen.")

---

### TensorBoard starten

Die finalen Trainingslogs werden unter `training/logs/final` gespeichert. TensorBoard kann genutzt werden, um die Trainingsverläufe der finalen Runs zu prüfen.

Dieser Schritt dient nur zur Kontrolle des Trainingsverlaufs. Die finale Evaluation erfolgt später separat in Notebook 06

In [ ]:
tensorboard_command = "tensorboard --logdir training/logs/final"

print("TensorBoard Command:")
print(tensorboard_command)